# 06 Baseline Models

This notebook trains the structured-data baselines used as reference points for the multimodal study. It evaluates Logistic Regression, Random Forest, and XGBoost across the 6h, 12h, and 24h prediction horizons using the leakage-safe datasets from feature engineering.

## Baseline strategy

- Convert each horizon trajectory dataset into one stay-level tabular example.
- Aggregate temporal features with `mean`, `min`, `max`, and `last`.
- Train only on the train split and report validation/test metrics separately.
- Save both metrics and per-stay prediction tables for downstream comparison.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost

In [ ]:
import pandas as pd

from src.evaluation.metrics import compute_binary_classification_metrics
from src.models.baselines import (
    build_baseline_models,
    fit_and_predict_baseline,
    make_stay_level_tabular_dataset,
    split_tabular_dataset,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config['baselines']

## Load horizon datasets

In [ ]:
horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'])

{key: value.shape for key, value in horizon_tables.items()}

## Train baselines horizon-by-horizon

In [ ]:
models = build_baseline_models(config)
results_rows = []
artifact_bundle = {}

for dataset_name, horizon_df in horizon_tables.items():
    tabular_df = make_stay_level_tabular_dataset(
        horizon_df=horizon_df,
        aggregations=config['baselines']['tabular_aggregations'],
    )
    artifact_bundle[f'{dataset_name}_tabular'] = tabular_df
    splits = split_tabular_dataset(tabular_df)
    train_X, train_y = splits['train']
    val_X, val_y = splits['val']
    test_X, test_y = splits['test']

    for model_name, model in models.items():
        if train_X.empty or test_X.empty:
            continue

        test_prob, feature_cols = fit_and_predict_baseline(model, train_X, train_y, test_X)
        test_metrics = compute_binary_classification_metrics(test_y, test_prob)
        results_rows.append({
            'dataset_name': dataset_name,
            'split': 'test',
            'model_name': model_name,
            **test_metrics,
            'n_features': len(feature_cols),
            'n_examples': int(len(test_y)),
        })

        test_predictions = test_X[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID']].copy()
        test_predictions['y_true'] = test_y.to_numpy()
        test_predictions['y_prob'] = test_prob
        test_predictions['dataset_name'] = dataset_name
        test_predictions['model_name'] = model_name
        artifact_bundle[f'{dataset_name}_{model_name}_test_predictions'] = test_predictions

        if not val_X.empty:
            val_prob, _ = fit_and_predict_baseline(model, train_X, train_y, val_X)
            val_metrics = compute_binary_classification_metrics(val_y, val_prob)
            results_rows.append({
                'dataset_name': dataset_name,
                'split': 'val',
                'model_name': model_name,
                **val_metrics,
                'n_features': len(feature_cols),
                'n_examples': int(len(val_y)),
            })

results_df = pd.DataFrame(results_rows).sort_values(['dataset_name', 'split', 'model_name']).reset_index(drop=True)
results_df

## Inspect best-performing baselines

In [ ]:
results_df.sort_values(['split', 'auprc'], ascending=[True, False]).head(12) if not results_df.empty else results_df

## Save metrics and prediction artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '06_baseline_models'
artifact_bundle['baseline_results'] = results_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '06_baseline_models_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='06_baseline_models',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'models_run': list(models.keys()),
        'result_rows': int(len(results_df)),
    },
)
manifest_path

## Review checklist

Before moving to multimodal models, review:

- Which baseline is strongest at each horizon?
- Does AUPRC drop sharply from 6h to 24h?
- Are validation and test results reasonably aligned?
- Does XGBoost materially outperform simpler models?
- Are any horizons too imbalanced for stable comparison?

## Next notebook

`07_multimodal_models.ipynb` will introduce the deep structured encoders, text encoders, and fusion strategies that should be compared against these baselines.